In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from lib.helpers import get_run_date
from lib.job_control import get_max_insert_ts, insert_log
from pyspark.sql.functions import current_timestamp, lit, to_date
from lib.aws_s3 import s3_bucket_name, archive_landing_object
from lib.aws_s3 import warehouse_dir

In [ ]:
# Create the SparkSession
"""
spark = (
    SparkSession.builder
    .appName("03 - Dealer Landing")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)

spark
"""

In [ ]:
# Job parameters
now_ts = spark.range(1).select(current_timestamp().alias("now")).first()["now"]
run_date = get_run_date()
table = "dealer"
schema_name = "lnd"
table_name = f"dim_{table}_{schema_name}"
table_full_name = f"{schema_name}.{table_name}" 
landing_file_name = f"{table}_{run_date}.csv"
landing_file_path = f"s3a://{s3_bucket_name}/pyspark-dw-aws/landing/{table}/{landing_file_name}"

print("****************************************")
print("03 - Dealer Landing")
print("****************************************")
print(f"now_ts = {now_ts}")
print(f"run_date = {run_date}")
print(f"table = {table}")
print(f"schema_name = {schema_name}")
print(f"table_name = {table_name}")
print(f"table_full_name = {table_full_name}")
print(f"landing_file_name = {landing_file_name}")
print(f"landing_file_path = {landing_file_path}")
print("")

In [ ]:
# Get max insert_ts for {schema_name}.{table_name} table from prd.job_control
max_insert_ts = get_max_insert_ts(spark, schema_name, table_name)
print(f"Max insert_ts in {table_full_name}: {max_insert_ts}\n")

In [ ]:
# Read landing file and store in a dataframe
df_raw = (
    spark
    .read
    .format("csv")
    .option("header", True)
    .load(landing_file_path)
)

In [ ]:
print(f"df_raw: {df_raw.count()} records read from {landing_file_name}")
df_raw.show()
df_raw.printSchema()

In [ ]:
# Add audit columns insert_dt and run_date
df_lnd = (
    df_raw
    .withColumn("insert_ts", lit(now_ts))
    .withColumn("run_date", to_date(lit(run_date), "yyyyMMdd"))
)

In [ ]:
print("Added audit columns")
print(f"df_lnd: {df_lnd.count()} records")
df_lnd.show()
df_lnd.printSchema()

In [ ]:
# Load data to lnd schema
write_mode = "append" if get_max_insert_ts(spark, schema_name, table_name) != "1900-01-01 00:00:00.000000" else "overwrite"

(
    df_lnd.write
    .format("delta")
    .mode(write_mode)
    .saveAsTable(table_full_name)
)

print(f"{df_lnd.count()} records written to {table_full_name} (mode={write_mode})")

In [ ]:
print(f"Data loaded in this run:")
spark.table(f'{schema_name}.dim_{table}_{schema_name}').where(f"insert_ts = '{now_ts}'").show()

In [ ]:
# Archive landing file
if archive_landing_object(landing_file_name, table):
    print(f"{landing_file_name} archived!")
else:
    print(f"{landing_file_name} not archived!")

In [ ]:
# Add entry to prd.job_control
if insert_log(spark, schema_name, table_name, now_ts, run_date):
    print(f"{schema_name}.{table_name} entry added to prd.job_control")

In [ ]:
# Review added entry
spark.sql(f"select * from prd.job_control where table_name = '{table_name}' order by insert_ts desc limit 1").show(truncate=False)

In [ ]:
# Stop spark session
#spark.stop()